In [ ]:
# Install necessary libraries
!pip install encodec torchaudio librosa matplotlib
import torch
import torchaudio
import librosa
import time
import numpy as np
import matplotlib.pyplot as plt
from encodec import EncodecModel
from encodec.utils import convert_audio

In [ ]:
# Model A - Traditional DSP (Mel-Quantization)
def encrypt_dsp(audio_path):
    start_time = time.time()

    y, sr = librosa.load(audio_path, sr=22050)

    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel_spec, ref=np.max)

    tokens = np.round(mel_db).astype(int)

    runtime = time.time() - start_time
    return tokens, runtime

def decrypt_dsp(tokens):
    start_time = time.time()

    mel_db = tokens.astype(float)
    mel_spec = librosa.db_to_power(mel_db)

    y_inv = librosa.feature.inverse.mel_to_audio(mel_spec)

    runtime = time.time() - start_time
    return y_inv, runtime

In [ ]:
# Model B - Neural Codec (RVQ Encryption)
def encrypt_neural(audio_path):
    model = EncodecModel.encodec_model_24khz()
    model.set_target_bandwidth(6.0)

    start_time = time.time()

    y, sr = librosa.load(audio_path, sr=None)

    wav = torch.from_numpy(y).unsqueeze(0)

    wav = convert_audio(wav, sr, model.sample_rate, model.channels)
    wav = wav.unsqueeze(0)

    with torch.no_grad():
        frames = model.encode(wav)
    tokens = frames[0][0]

    runtime = time.time() - start_time
    return tokens, runtime, model

def decrypt_neural(tokens, model):
    start_time = time.time()
    with torch.no_grad():
        frames = [(tokens, None)]
        out_wav = model.decode(frames)
    runtime = time.time() - start_time
    return out_wav.squeeze(0).cpu().numpy(), runtime

In [ ]:
# Model C - Chaotic Signal Masking
def logistic_map(seed, n, r=3.99):
    x = np.zeros(n)
    x[0] = seed
    for i in range(1, n):
        x[i] = r * x[i-1] * (1 - x[i-1])
    return (x - 0.5) * 2

def encrypt_chaos(audio_path, key_seed=0.5, intensity=0.8):
    start_time = time.time()

    y, sr = librosa.load(audio_path, sr=22050)

    chaos_key = logistic_map(key_seed, len(y))

    y_encrypted = y + (chaos_key * intensity)

    runtime = time.time() - start_time
    return y_encrypted, runtime, chaos_key

def decrypt_chaos(y_encrypted, chaos_key, intensity=0.8):
    start_time = time.time()

    y_decrypted = y_encrypted - (chaos_key * intensity)

    runtime = time.time() - start_time
    return y_decrypted, runtime

test_file = "harvard.wav"

audio_enc_c, time_c, key_c = encrypt_chaos(test_file)
audio_dec_c, time_dec_c = decrypt_chaos(audio_enc_c, key_c)


In [ ]:
test_file = "harvard.wav"

tokens_a, time_a = encrypt_dsp(test_file)
tokens_b, time_b, model_b = encrypt_neural(test_file)
tokens_c, time_c, keys_c = encrypt_chaos(test_file)

print(f"Model A (DSP)    | Runtime: {time_a:.4f}s")
print(f"Model B (Neural) | Runtime: {time_b:.4f}s")
print(f"Model C (Chaos)| Runtime: {time_c:.4f}s")

# Final Visualization
plt.figure(figsize=(15, 8))

# A
plt.subplot(3, 1, 1);
plt.title("Model A: DSP (No Privacy)");
plt.imshow(tokens_a, aspect='auto')

# B
plt.subplot(3, 1, 2);
plt.title("Model B: Neural (Content Shielded)");
plt.imshow(tokens_b[0, 0:1, :].cpu().numpy(), aspect='auto')

# C
S = np.abs(librosa.stft(tokens_c))
plt.subplot(3, 1, 3)
plt.title("Model C: Chaotic Masking (InaudibleKey)")
plt.imshow(librosa.amplitude_to_db(S, ref=np.max), aspect='auto', origin='lower')

plt.tight_layout()
plt.show()